# Missing ModernMolBERT benchmark audit and optional rerun

This notebook checks whether each benchmark dataset is present for each ModernMolBERT model in `outputs/eval/best_metric_by_dataset_embedder.csv`. It also distinguishes three useful intermediate states: rows present in the raw per-model `results.csv`, rows present only in per-dataset checkpoints, and missing embedding files.

Manuscript note: the current manuscript is `paper/main.tex`. Its title and placeholder abstract frame the primary manuscript contribution as a ModernBERT encoder family for SELFIES molecular language modeling. For the internal ablation axis, `SPAN_MASKING_PLAN_V2.md` defines the ladder as `masking_strategy`: `standard` vs `span` vs `hetero_span`. The `base` model is a capacity/scale comparison, not the masking-strategy ablation axis.

In [1]:
from pathlib import Path
import shlex
import subprocess

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/modernmolbert").exists():
            return path
    raise RuntimeError("Could not locate ModernMolBERT repo root")


ROOT = find_repo_root()
BEST_CSV = ROOT / "outputs/eval/best_metric_by_dataset_embedder.csv"
EVAL_ROOT = ROOT / "outputs/eval"
DATASET_CONFIG = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/config/datasets.yaml"
SCORE_PY = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/score.py"
EMBED_PY = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/embed_modernmolbert.py"
BUILD_BENCHMARK_FRAMES_PY = ROOT / "scripts/build_benchmark_results_frames.py"

MODELS = [
    "modernmolbert_best_standard",
    "modernmolbert_best_base",
    "modernmolbert_best_span",
    "modernmolbert_best_hetero_span",
]

MODEL_TO_RESULT_DIR = {
    "modernmolbert_best_standard": EVAL_ROOT / "praski_best_standard",
    "modernmolbert_best_base": EVAL_ROOT / "praski_best_base_standard",
    "modernmolbert_best_span": EVAL_ROOT / "praski_best_span",
    "modernmolbert_best_hetero_span": EVAL_ROOT / "praski_best_hetero_span",
}

# Used only when RUN_MISSING_EMBEDDINGS is enabled below.
# NOTE: _standard and _span have weights under final_model/; _hetero_span has
# weights directly in the run dir; _base also has weights under final_model/.
MODEL_TO_MODEL_DIR = {
    "modernmolbert_best_standard": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_standard/final_model",
    "modernmolbert_best_base": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_base/final_model",
    "modernmolbert_best_span": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_span/final_model",
    "modernmolbert_best_hetero_span": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_hetero_span",
}

print(ROOT)

/Users/skn506/Documents/ModernMolBERT


In [2]:
HEADS = ["rf", "ridge", "knn"]


def load_best() -> pd.DataFrame:
    if not BEST_CSV.exists():
        raise FileNotFoundError(BEST_CSV)
    return pd.read_csv(BEST_CSV)


def safe_component(value: str) -> str:
    return str(value).replace("/", "_").replace("\\", "_").replace(":", "_").replace(" ", "_")


def result_csv(model: str) -> Path:
    return MODEL_TO_RESULT_DIR[model] / "results.csv"


def checkpoint_path(dataset: str, model: str) -> Path:
    return (
        MODEL_TO_RESULT_DIR[model]
        / "checkpoints"
        / f"{safe_component(dataset)}__{safe_component(model)}.csv"
    )


def embedding_path(dataset: str, model: str) -> Path:
    return ROOT / "data/embedded" / dataset / f"{model}.joblib"


def dataset_selector(dataset: str) -> str:
    selector = f"clf_{dataset}"
    text = DATASET_CONFIG.read_text()
    if f"  {selector}:" in text:
        return selector
    if f"  {dataset}:" in text:
        return dataset
    raise KeyError(f"No dataset selector found for {dataset!r} in {DATASET_CONFIG}")


def csv_has_pair(path: Path, dataset: str, model: str) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        df = pd.read_csv(path, usecols=["dataset", "embedder"])
    except ValueError:
        df = pd.read_csv(path)
    if not {"dataset", "embedder"}.issubset(df.columns):
        return False
    return bool(((df["dataset"] == dataset) & (df["embedder"] == model)).any())


def heads_in_csv(path: Path, dataset: str, model: str) -> frozenset[str]:
    """Return the set of scorer heads already present in a results CSV."""
    if not path.exists() or path.stat().st_size == 0:
        return frozenset()
    try:
        df = pd.read_csv(path, usecols=["dataset", "embedder", "model"])
    except ValueError:
        return frozenset()
    if not {"dataset", "embedder", "model"}.issubset(df.columns):
        return frozenset()
    mask = (df["dataset"] == dataset) & (df["embedder"] == model)
    return frozenset(df.loc[mask, "model"].dropna().unique())


def benchmark_present(dataset: str, model: str, *, source: str = "aggregate") -> bool:
    if source == "aggregate":
        df = load_best()
        return bool(((df["dataset"] == dataset) & (df["embedder"] == model)).any())
    if source == "raw":
        return csv_has_pair(result_csv(model), dataset, model)
    if source == "checkpoint":
        return csv_has_pair(checkpoint_path(dataset, model), dataset, model)
    raise ValueError(f"Unknown source: {source}")


def audit_coverage() -> pd.DataFrame:
    best = load_best()
    datasets = sorted(best["dataset"].dropna().unique())
    records = []
    for model in MODELS:
        raw_path = result_csv(model)
        for dataset in datasets:
            in_aggregate = benchmark_present(dataset, model, source="aggregate")
            in_raw = benchmark_present(dataset, model, source="raw")
            in_checkpoint = benchmark_present(dataset, model, source="checkpoint")
            has_embedding = embedding_path(dataset, model).exists()

            # Per-head coverage from raw results.csv and checkpoint
            raw_heads = heads_in_csv(raw_path, dataset, model)
            ckpt_heads = heads_in_csv(checkpoint_path(dataset, model), dataset, model)
            present_heads = raw_heads | ckpt_heads
            missing_heads = frozenset(HEADS) - present_heads

            if in_aggregate:
                status = "present"
            elif in_raw or in_checkpoint:
                status = "result_not_in_aggregate"
            elif has_embedding:
                status = "missing_score_can_rerun"
            else:
                status = "missing_embedding"
            records.append(
                {
                    "dataset": dataset,
                    "model": model,
                    "in_aggregate": in_aggregate,
                    "in_raw_results": in_raw,
                    "in_checkpoint": in_checkpoint,
                    "has_embedding": has_embedding,
                    "status": status,
                    "heads_present": sorted(present_heads) or None,
                    "missing_heads": sorted(missing_heads) if missing_heads else None,
                    "dataset_selector": dataset_selector(dataset),
                    "result_csv": result_csv(model).relative_to(ROOT),
                    "checkpoint_csv": checkpoint_path(dataset, model).relative_to(ROOT),
                    "embedding_file": embedding_path(dataset, model).relative_to(ROOT),
                }
            )
    return pd.DataFrame.from_records(records)


coverage = audit_coverage()
display(coverage.groupby(["model", "status"]).size().unstack(fill_value=0))
display(coverage[coverage["status"] != "present"].sort_values(["model", "dataset"]))

status,missing_embedding,missing_score_can_rerun,present,result_not_in_aggregate
model,,,,
modernmolbert_best_base,2,0,24,0
modernmolbert_best_hetero_span,4,0,22,0
modernmolbert_best_span,0,2,23,1
modernmolbert_best_standard,0,0,26,0


,dataset,model,in_aggregate,in_raw_results,in_checkpoint,has_embedding,status,heads_present,missing_heads,dataset_selector,result_csv,checkpoint_csv,embedding_file
50,ogbg-moltox21,modernmolbert_best_base,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-moltox21,outputs/eval/praski_best_base_standard/results...,outputs/eval/praski_best_base_standard/checkpo...,data/embedded/ogbg-moltox21/modernmolbert_best...
51,ogbg-moltoxcast,modernmolbert_best_base,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-moltoxcast,outputs/eval/praski_best_base_standard/results...,outputs/eval/praski_best_base_standard/checkpo...,data/embedded/ogbg-moltoxcast/modernmolbert_be...
100,ogbg-molmuv,modernmolbert_best_hetero_span,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-molmuv,outputs/eval/praski_best_hetero_span/results.csv,outputs/eval/praski_best_hetero_span/checkpoin...,data/embedded/ogbg-molmuv/modernmolbert_best_h...
101,ogbg-molsider,modernmolbert_best_hetero_span,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-molsider,outputs/eval/praski_best_hetero_span/results.csv,outputs/eval/praski_best_hetero_span/checkpoin...,data/embedded/ogbg-molsider/modernmolbert_best...
102,ogbg-moltox21,modernmolbert_best_hetero_span,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-moltox21,outputs/eval/praski_best_hetero_span/results.csv,outputs/eval/praski_best_hetero_span/checkpoin...,data/embedded/ogbg-moltox21/modernmolbert_best...
103,ogbg-moltoxcast,modernmolbert_best_hetero_span,False,False,False,False,missing_embedding,None,"[knn, rf, ridge]",clf_ogbg-moltoxcast,outputs/eval/praski_best_hetero_span/results.csv,outputs/eval/praski_best_hetero_span/checkpoin...,data/embedded/ogbg-moltoxcast/modernmolbert_be...
73,ogbg-molhiv,modernmolbert_best_span,False,False,True,True,result_not_in_aggregate,"[knn, rf, ridge]",None,clf_ogbg-molhiv,outputs/eval/praski_best_span/results.csv,outputs/eval/praski_best_span/checkpoints/ogbg...,data/embedded/ogbg-molhiv/modernmolbert_best_s...
74,ogbg-molmuv,modernmolbert_best_span,False,False,False,True,missing_score_can_rerun,None,"[knn, rf, ridge]",clf_ogbg-molmuv,outputs/eval/praski_best_span/results.csv,outputs/eval/praski_best_span/checkpoints/ogbg...,data/embedded/ogbg-molmuv/modernmolbert_best_s...
77,ogbg-moltoxcast,modernmolbert_best_span,False,False,False,True,missing_score_can_rerun,None,"[knn, rf, ridge]",clf_ogbg-moltoxcast,outputs/eval/praski_best_span/results.csv,outputs/eval/praski_best_span/checkpoints/ogbg...,data/embedded/ogbg-moltoxcast/modernmolbert_be...


In [3]:
coverage_matrix = coverage.pivot(index="dataset", columns="model", values="status")
display(coverage_matrix[MODELS])

missing_by_model = (
    coverage.loc[~coverage["in_aggregate"]]
    .groupby("model")["dataset"]
    .apply(list)
    .reindex(MODELS, fill_value=[])
)

for model, datasets in missing_by_model.items():
    print(f"{model}: {26 - len(datasets)}/26 present; missing={datasets}")

model,modernmolbert_best_standard,modernmolbert_best_base,modernmolbert_best_span,modernmolbert_best_hetero_span
dataset,,,,
AMES,present,present,present,present
Bioavailability_Ma,present,present,present,present
CYP1A2_Veith,present,present,present,present
CYP2C19_Veith,present,present,present,present
CYP2C9_Substrate_CarbonMangels,present,present,present,present
CYP2C9_Veith,present,present,present,present
CYP2D6_Substrate_CarbonMangels,present,present,present,present
CYP2D6_Veith,present,present,present,present
CYP3A4_Substrate_CarbonMangels,present,present,present,present


modernmolbert_best_standard: 26/26 present; missing=[]
modernmolbert_best_base: 24/26 present; missing=['ogbg-moltox21', 'ogbg-moltoxcast']
modernmolbert_best_span: 23/26 present; missing=['ogbg-molhiv', 'ogbg-molmuv', 'ogbg-moltoxcast']
modernmolbert_best_hetero_span: 22/26 present; missing=['ogbg-molmuv', 'ogbg-molsider', 'ogbg-moltox21', 'ogbg-moltoxcast']


In [4]:
def list_missing(coverage: pd.DataFrame) -> None:
    """Print every missing (dataset, model, head) triple, grouped by model.

    Also prints copy-paste-ready RERUN_TARGETS entries for use in the cell below.
    """
    missing = coverage.loc[~coverage["in_aggregate"]].copy()

    if missing.empty:
        print("No missing evals — all datasets present in aggregate CSV.")
        return

    print(f"{'=' * 70}")
    print(f"MISSING EVALS  ({len(missing)} dataset/model pairs across all heads)")
    print(f"{'=' * 70}")

    rerun_lines: list[str] = []

    for model, group in missing.groupby("model", sort=False):
        print(f"\n  {model}  ({len(group)} datasets)")
        for row in group.sort_values("dataset").itertuples(index=False):
            missing_h = row.missing_heads if row.missing_heads is not None else list(HEADS)
            present_h = row.heads_present if row.heads_present is not None else []
            status_tag = f"[{row.status}]"
            print(f"    {row.dataset:<35} present={present_h}  missing={missing_h}  {status_tag}")

            if len(missing_h) == len(HEADS):
                # All heads missing — emit a 2-tuple
                rerun_lines.append(f'    ("{row.dataset}", "{model}"),')
            else:
                # Only specific heads missing — emit one 3-tuple per head
                for h in missing_h:
                    rerun_lines.append(f'    ("{row.dataset}", "{model}", "{h}"),')

    print(f"\n{'=' * 70}")
    print("RERUN_TARGETS snippet (copy into the cell below):")
    print("RERUN_TARGETS = [")
    for line in rerun_lines:
        print(line)
    print("]")


list_missing(coverage)

MISSING EVALS  (9 dataset/model pairs across all heads)

  modernmolbert_best_base  (2 datasets)
    ogbg-moltox21                       present=[]  missing=['knn', 'rf', 'ridge']  [missing_embedding]
    ogbg-moltoxcast                     present=[]  missing=['knn', 'rf', 'ridge']  [missing_embedding]

  modernmolbert_best_span  (3 datasets)
    ogbg-molhiv                         present=['knn', 'rf', 'ridge']  missing=['rf', 'ridge', 'knn']  [result_not_in_aggregate]
    ogbg-molmuv                         present=[]  missing=['knn', 'rf', 'ridge']  [missing_score_can_rerun]
    ogbg-moltoxcast                     present=[]  missing=['knn', 'rf', 'ridge']  [missing_score_can_rerun]

  modernmolbert_best_hetero_span  (4 datasets)
    ogbg-molmuv                         present=[]  missing=['knn', 'rf', 'ridge']  [missing_embedding]
    ogbg-molsider                       present=[]  missing=['knn', 'rf', 'ridge']  [missing_embedding]
    ogbg-moltox21                       present=

## Optional rerun controls

Default behavior is audit-only. Flip the relevant booleans to run subprocesses. Scoring uses `--no-resume` so an existing checkpoint cannot cause a missing aggregate row to be skipped silently. After any scoring run, rebuild the aggregate CSV with `scripts/build_benchmark_results_frames.py`.

In [ ]:
# Execution toggles. Keep these False for audit-only runs.
RUN_MISSING_EMBEDDINGS = True
RUN_MISSING_SCORE = False
REBUILD_AGGREGATES_AFTER_SCORE = False

# Choose targets. Options:
#   "all_missing"                            — every non-aggregate row
#   [(dataset, model), ...]                  — specific dataset/model pairs; all heads scored
#   [(dataset, model, head), ...]            — specific dataset/model/head triples (passes --heads to scorer)
#
# Examples:
#   RERUN_TARGETS = [("ogbg-molhiv", "modernmolbert_best_span")]
#   RERUN_TARGETS = [("AMES", "modernmolbert_best_base", "knn")]
RERUN_TARGETS: str | list[tuple] = [("ogbg-moltox21", "modernmolbert_best_base")]  # "all_missing"

# Embedding settings used only when RUN_MISSING_EMBEDDINGS is True.
EMBED_BATCH_SIZE = 32
EMBED_DEVICE = "auto"
EMBED_MAX_SEQ_LENGTH = 256
EMBED_POOLING = "mean"
OVERWRITE_EXISTING_EMBEDDINGS = False


def selected_targets(coverage: pd.DataFrame) -> pd.DataFrame:
    missing = coverage.loc[~coverage["in_aggregate"]].copy()
    missing["target_heads"] = [list(HEADS)] * len(missing)

    if RERUN_TARGETS == "all_missing":
        return missing

    rows = []
    for entry in RERUN_TARGETS:
        if len(entry) == 2:
            dataset, model = entry
            head_filter = list(HEADS)
        elif len(entry) == 3:
            dataset, model, head = entry
            head_filter = [head]
        else:
            raise ValueError(f"RERUN_TARGETS entries must be 2- or 3-tuples, got: {entry!r}")
        mask = (missing["dataset"] == dataset) & (missing["model"] == model)
        subset = missing.loc[mask].copy()
        subset["target_heads"] = [head_filter] * len(subset)
        rows.append(subset)

    return pd.concat(rows, ignore_index=True) if rows else missing.iloc[:0].copy()


targets = selected_targets(coverage)
targets_to_embed = targets.loc[~targets["has_embedding"]].copy()
targets_to_score = targets.loc[targets["has_embedding"]].copy()

print(f"selected missing targets: {len(targets)}")
print(f"targets needing embeddings first: {len(targets_to_embed)}")
print(f"targets ready to score: {len(targets_to_score)}")
display(
    targets[
        ["dataset", "model", "status", "heads_present", "missing_heads", "target_heads"]
    ].sort_values(["model", "dataset"])
)

In [ ]:
def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def score_command(dataset: str, model: str, heads: list[str] | None = None) -> list[str]:
    out_dir = MODEL_TO_RESULT_DIR[model]
    cmd = [
        "uv",
        "run",
        "python",
        rel(SCORE_PY),
        "--datasets",
        dataset_selector(dataset),
        "--embedder",
        model,
        "--output-csv",
        rel(out_dir / "results.csv"),
        "--checkpoint-dir",
        rel(out_dir / "checkpoints"),
        "--no-resume",
        "--safe",
    ]
    if heads and set(heads) != set(HEADS):
        cmd += ["--heads"] + heads
    return cmd


def embed_command(dataset: str, model: str) -> list[str]:
    model_dir = MODEL_TO_MODEL_DIR[model]
    cmd = [
        "uv",
        "run",
        "python",
        rel(EMBED_PY),
        "--datasets",
        dataset_selector(dataset),
        "--model-dir",
        rel(model_dir),
        "--tokenizer-path",
        rel(model_dir),
        "--embedder",
        model,
        "--batch-size",
        str(EMBED_BATCH_SIZE),
        "--device",
        EMBED_DEVICE,
        "--max-seq-length",
        str(EMBED_MAX_SEQ_LENGTH),
        "--pooling",
        EMBED_POOLING,
    ]
    if OVERWRITE_EXISTING_EMBEDDINGS:
        cmd.append("--overwrite")
    return cmd


def run_command(cmd: list[str], *, enabled: bool) -> None:
    printable = " ".join(shlex.quote(part) for part in cmd)
    print("$", printable)
    if enabled:
        subprocess.run(cmd, cwd=ROOT, check=True)


print("Embedding commands for missing embeddings:")
for row in targets_to_embed.sort_values(["model", "dataset"]).itertuples(index=False):
    run_command(embed_command(row.dataset, row.model), enabled=False)

print("\nScoring commands for targets with embeddings:")
for row in targets_to_score.sort_values(["model", "dataset"]).itertuples(index=False):
    run_command(score_command(row.dataset, row.model, row.target_heads), enabled=False)

In [ ]:
if RUN_MISSING_EMBEDDINGS:
    for row in targets_to_embed.sort_values(["model", "dataset"]).itertuples(index=False):
        model_dir = MODEL_TO_MODEL_DIR[row.model]
        if not model_dir.exists():
            raise FileNotFoundError(f"Model directory does not exist: {model_dir}")
        run_command(embed_command(row.dataset, row.model), enabled=True)

    coverage = audit_coverage()
    targets = selected_targets(coverage)
    targets_to_score = targets.loc[targets["has_embedding"]].copy()

if RUN_MISSING_SCORE:
    for row in targets_to_score.sort_values(["model", "dataset"]).itertuples(index=False):
        run_command(score_command(row.dataset, row.model, row.target_heads), enabled=True)

if REBUILD_AGGREGATES_AFTER_SCORE:
    run_command(["uv", "run", "python", rel(BUILD_BENCHMARK_FRAMES_PY)], enabled=True)

In [ ]:
# Re-run this cell after optional scoring and aggregate rebuild.
coverage_after = audit_coverage()
display(coverage_after.groupby(["model", "status"]).size().unstack(fill_value=0))
display(coverage_after.loc[~coverage_after["in_aggregate"]].sort_values(["model", "dataset"]))